In [ ]:
# Check biomass production of iMMU1867 under the defined minimal medium
# Model: iMMU1867 - A curated mouse genome-scale metabolic model reconstructed from iMM1865

# The defined minimal medium was formulated from basic biomass-supporting nutrients, including glucose, oxygen,
# inorganic ions, phosphate, ammonium, and essential amino acids. Uptake bounds were parameterized to approximate
# previously reported astrocyte biomass production of ~0.32 h^1.

# Objective function: BIOMASS_reaction
# ------------------------------------------------------------

import cobra
from cobra.io import read_sbml_model
from pathlib import Path

# Load model
# Path relative to the repository root
MODEL_PATH = Path.cwd().parents[1] / "models" / "02_curated_iMMU1867" / "iMMU1867.xml"
BIOMASS_ID = "BIOMASS_reaction"

print("Model loading.....")
model = read_sbml_model(str(MODEL_PATH))

# Check model composition
print(f"Model Composition: {len(model.reactions)} reactions, "
      f"{len(model.metabolites)} metabolites, {len(model.genes)} genes")

# Set objective function - Biomass reaction
model.objective = BIOMASS_ID

Model loading.....
Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2685367
Academic license 2685367 - for non-commercial use only - registered to sa___@wsu.edu
Model Composition: 10436 reactions, 5873 metabolites, 1867 genes


In [ ]:
# Run FBA before applying the defined medium

solution = model.optimize()
print(f"Biomass objective value: {solution.objective_value:.4f}/h")

Biomass objective value: 0.3269/h


In [5]:
# Defined iMMU1867 medium

# Bounds are given as (lower_bound, upper_bound).
# Negative lower bounds allow uptake.

DEFINED_MEDIUM = {

    # Major ions and solvent
    "EX_ca2_e": (-1000, 1000),   # Calcium
    "EX_cl_e": (-1000, 1000),    # Chloride
    "EX_h2o_e": (-1000, 1000),   # Water
    "EX_h_e":   (-1000, 1000),   # Proton
    "EX_k_e":   (-1000, 1000),   # Potassium
    "EX_na1_e": (-1000, 1000),   # Sodium

    # Primary carbon source
    "EX_glc__D_e": (-10, 1000),  # D-glucose

    # Inorganic nutrients and gases
    "EX_nh4_e": (-10, 1000),     # Ammonium
    "EX_o2_e":  (-10, 1000),     # Oxygen
    "EX_pi_e":  (-10, 1000),     # Phosphate
    "EX_so4_e": (-10, 1000),     # Sulfate

    # Essential amino acids
    "EX_his__L_e": (-0.041322, 1000),
    "EX_ile__L_e": (-0.093519, 1000),
    "EX_leu__L_e": (-0.178338, 1000),
    "EX_lys__L_e": (-0.193562, 1000),
    "EX_met__L_e": (-0.065246, 1000),
    "EX_phe__L_e": (-0.137016, 1000),
    "EX_thr__L_e": (-0.102218, 1000),
    "EX_trp__L_e": (-0.004350, 1000),
    "EX_val__L_e": (-0.115267, 1000),

}

In [ ]:
def close_all_exchanges(model):
    # Close all exchange uptake reactions before applying the defined medium.
    for rxn in model.exchanges:
        rxn.lower_bound = 0
        rxn.upper_bound = 1000


def apply_defined_medium(model, medium):
    #Apply defined medium bounds to the model.
    close_all_exchanges(model)

    missing_reactions = []

    for rxn_id, (lower_bound, upper_bound) in medium.items():
        if rxn_id in model.reactions:
            rxn = model.reactions.get_by_id(rxn_id)
            rxn.lower_bound = lower_bound
            rxn.upper_bound = upper_bound
        else:
            missing_reactions.append(rxn_id)

    return missing_reactions

In [9]:
# Run FBA after applying the defined medium

print("Running FBA under defined minimal medium.....")

with model:
    missing = apply_defined_medium(model, DEFINED_MEDIUM)

    if missing:
        print("These medium reactions were not found in the model:")
        for rxn_id in missing:
            print(f"  - {rxn_id}")

    solution = model.optimize()

    print(f"\nSolver status: {solution.status}")

    if solution.status == "optimal":
        biomass_flux = solution.fluxes[BIOMASS_ID]
        print(f"Biomass objective value: {biomass_flux:.4f}/h")
    else:
        print("No optimal solution found.")

Running FBA under defined minimal medium.....

Solver status: optimal
Biomass objective value: 0.3269/h
